# Model 1: Training & Evaluation
## Stratified K-Fold Cross-Validation

In [16]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score, f1_score, roc_curve, precision_recall_curve
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: mps


## Hyperparameters 

Hyperparameters were set to these to maximize sensitivity

In [17]:
N_FOLDS       = 5
BATCH_SIZE    = 64
LEARNING_RATE = 5e-4
NUM_EPOCHS    = 40
MIN_SENSITIVITY = 0.8

Loaded the arrays for the train, test, and patient ids (from pre-processed data)

In [18]:
X_train           = np.load('X_train.npy')
X_test            = np.load('X_test.npy')
y_train           = np.load('y_train.npy')
y_test            = np.load('y_test.npy')
train_patient_ids = np.load('train_patient_ids.npy')
test_patient_ids  = np.load('test_patient_ids.npy')
train_locations = np.load('train_locations.npy', allow_pickle=True)
test_locations  = np.load('test_locations.npy', allow_pickle=True)

print(f'X_train shape: {X_train.shape}')
print(f'y_train — Murmur present: {y_train.sum()}, Absent: {(y_train==0).sum()}')

X_train shape: (46733, 4000)
y_train — Murmur present: 9963, Absent: 36770


In [19]:
print(train_locations[:10])
print(np.unique(train_locations))

['AV' 'AV' 'AV' 'AV' 'AV' 'AV' 'AV' 'AV' 'AV' 'AV']
['AV' 'MV' 'PV' 'TV']


Replicated the function and classes from model_1d_cnn here since cross‑notebook imports weren’t accessible.

In [20]:
def normalize_waveforms(X):
    mean = X.mean(axis=1, keepdims=True)
    std  = X.std(axis=1, keepdims=True)
    std  = np.where(std == 0, 1, std)
    return (X - mean) / std

X_train_norm = normalize_waveforms(X_train)
X_test_norm  = normalize_waveforms(X_test)

In [21]:
class HeartbeatDataset(Dataset):
    def __init__(self, X, y, patient_ids):
        self.X           = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        self.y           = torch.tensor(y, dtype=torch.float32)
        self.patient_ids = patient_ids

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.patient_ids[idx]

Removed the nn.Sigmoid from the MurmurCNN1D class (model1_1d_cnn) because BCEWithLogitsLoss handles sigmoid during training, but torch.sigmoid() was added in 'get_heartbeat_predictions''

In [22]:
class MurmurCNN1D(nn.Module):
    def __init__(self):
        super(MurmurCNN1D, self).__init__()
        self.block1 = nn.Sequential(
            nn.Conv1d(1,  32,  kernel_size=15, padding=7),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(4)
        )
        self.block2 = nn.Sequential(
            nn.Conv1d(32, 64,  kernel_size=9,  padding=4),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(4)
        )
        self.block3 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=5,  padding=2),
            nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(4)
        )
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.global_avg_pool(x)
        return self.classifier(x)

### Helper Functions

- `train_one_epoch` – Runs one full pass of training through the dataset. It handles forward and backward passes, updates the model weights, and keeps track of the average loss for that epoch.  

- `get_batch_predictions` – Loops through the dataloader to collect all predictions and their true labels

- `aggregate_by_patient_level` – Since our data is patch based, the function groups the predictions by patient and averages them so that we can measure performance at the patient level instead of just individual patches

- `compute_metrics_from_threshold` – Calculates metrics like sensitivity, specificity, and accuracy for a given probability threshold using the confusion matrix

In [23]:
def train_one_epoch(model, dataloader, criterion, optimizer):
    model.train()
    total_loss = 0
    for X_batch, y_batch, _ in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch).squeeze(1)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
    return total_loss / len(dataloader.dataset)

def get_heartbeat_predictions(model, dataloader):
    model.eval()
    all_outputs, all_labels, all_patient_ids = [], [], []
    with torch.no_grad():
        for X_batch, y_batch, patient_ids in dataloader:
            X_batch = X_batch.to(device)
            outputs = torch.sigmoid(model(X_batch)).squeeze().cpu().numpy()
            all_outputs.extend(outputs)
            all_labels.extend(y_batch.cpu().numpy())
            all_patient_ids.extend(patient_ids)
    return np.array(all_outputs), np.array(all_labels), np.array(all_patient_ids)

def aggregate_to_patient_level(preds, labels, patient_ids):
    df = pd.DataFrame({'patient_id': patient_ids, 'pred': preds, 'label': labels})
    patient_agg = df.groupby('patient_id').agg({
        'pred': 'max', #max probability across all heartbeats for that patient
        'label': 'max' #if any heartbeat is positive, patient is positive
        }).reset_index()
    return patient_agg

def compute_metrics(y_true, y_pred, threshold):
    auc = roc_auc_score(y_true, y_pred)
    auprc = average_precision_score(y_true, y_pred)

    # Convert probabilities to binary predictions using the threshold
    y_binary = (y_pred >= threshold).astype(int)
    f1  = f1_score(y_true, y_binary)

    tn, fp, fn, tp = confusion_matrix(y_true, y_binary).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    return {'auc': auc, 'auprc': auprc, 'f1': f1, 'sensitivity': sensitivity, 'specificity': specificity, 'threshold': threshold}

### Cross-Validation Training

This part of the code trains and tests the model using **Stratified K-Fold cross-validation**

The data is split into five folds, and the model is trained and tested on different parts each time to see how well it generalizes

- train_meta is a dataframe made to link each patient ID with its label, making sure each patient only appears once  

- The data is split into N_FOLDS groups while keeping the label balance the same in each fold  

- For each fold:
  - The code picks which patients go into training and which go into validation  
  - It creates masks to select the right samples for each group
  - It builds datasets and dataloaders for training and validation
  - A new model is created and trained for 40 epochs 
  - After training predictions are made on the validation set, and the AUROC score is calculated to measure performance  

In [ ]:
train_meta = pd.DataFrame({
    'patient_id': train_patient_ids,
    'label':      y_train
}).drop_duplicates(subset='patient_id')

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

cv_val_probs  = []
cv_val_labels = []
fold_metrics  = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train_meta, train_meta['label'])):
    print(f'\n Fold {fold+1}/{N_FOLDS} - Train patients: {len(train_idx)}, Val patients: {len(val_idx)}')

    train_pids = train_meta.iloc[train_idx]['patient_id'].values
    val_pids   = train_meta.iloc[val_idx]['patient_id'].values

    train_mask = np.isin(train_patient_ids, train_pids)
    val_mask   = np.isin(train_patient_ids, val_pids)

    fold_train_ds = HeartbeatDataset(X_train_norm[train_mask], y_train[train_mask], train_patient_ids[train_mask])
    fold_val_ds   = HeartbeatDataset(X_train_norm[val_mask],   y_train[val_mask],   train_patient_ids[val_mask])

    fold_train_loader = DataLoader(fold_train_ds, batch_size=BATCH_SIZE, shuffle=True)
    fold_val_loader   = DataLoader(fold_val_ds,   batch_size=BATCH_SIZE, shuffle=False)

    model          = MurmurCNN1D().to(device)
    fold_optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    pos_weight     = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()], dtype=torch.float32).to(device)
    criterion      = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    for epoch in range(NUM_EPOCHS):
        train_loss = train_one_epoch(model, fold_train_loader, criterion, fold_optimizer)
        if (epoch + 1) % 5 == 0:
            print(f'  Epoch {epoch+1}/{NUM_EPOCHS} — Loss: {train_loss:.4f}')
            
    val_probs, val_labels, val_pids_out = get_heartbeat_predictions(model, fold_val_loader)
    val_patient_df = aggregate_to_patient_level(val_probs, val_labels, val_pids_out)
    cv_val_probs.extend(val_patient_df['pred'].values)
    cv_val_labels.extend(val_patient_df['label'].values)
    fold_auroc = roc_auc_score(val_patient_df['label'], val_patient_df['pred'])
    fold_metrics.append({'fold': fold+1, 'auroc': fold_auroc})
    print(f'  Fold {fold+1} AUROC: {fold_auroc:.4f}')


 Fold 1/5 - Train patients: 556, Val patients: 140
  Epoch 5/40 — Loss: 0.8051
  Epoch 10/40 — Loss: 0.7353
  Epoch 15/40 — Loss: 0.6855
  Epoch 20/40 — Loss: 0.6429
  Epoch 25/40 — Loss: 0.6037
  Epoch 30/40 — Loss: 0.5659
  Epoch 35/40 — Loss: 0.5307
  Epoch 40/40 — Loss: 0.4843
  Fold 1 AUROC: 0.7353

 Fold 2/5 - Train patients: 557, Val patients: 139
  Epoch 5/40 — Loss: 0.8408
  Epoch 10/40 — Loss: 0.7765
  Epoch 15/40 — Loss: 0.7234
  Epoch 20/40 — Loss: 0.6892
  Epoch 25/40 — Loss: 0.6418
  Epoch 30/40 — Loss: 0.6057
  Epoch 35/40 — Loss: 0.5694
  Epoch 40/40 — Loss: 0.5291
  Fold 2 AUROC: 0.7896

 Fold 3/5 - Train patients: 557, Val patients: 139
  Epoch 5/40 — Loss: 0.8729
  Epoch 10/40 — Loss: 0.8133
  Epoch 15/40 — Loss: 0.7676
  Epoch 20/40 — Loss: 0.7385
  Epoch 25/40 — Loss: 0.6985
  Epoch 30/40 — Loss: 0.6571
  Epoch 35/40 — Loss: 0.6196
  Epoch 40/40 — Loss: 0.5914
  Fold 3 AUROC: 0.8552

 Fold 4/5 - Train patients: 557, Val patients: 139
  Epoch 5/40 — Loss: 0.8516
  

### Selecting the Best Classification Threshold

After collecting all validation predictions from cross-validation we find the best decision threshold for classifying samples.

- The ROC curve is used to get the FPR, TPR, and thresholds
- Only thresholds that reach at least the minimum required sensitivity are kept
- The code then picks the threshold with the smallest FPR, and that would be used for final eval

In [24]:
import os
cv_val_probs  = np.load('cv_val_probs.npy')
cv_val_labels = np.load('cv_val_labels.npy')

fpr, tpr, thresholds = roc_curve(cv_val_labels, cv_val_probs)
valid_mask       = tpr >= MIN_SENSITIVITY
valid_thresholds = thresholds[valid_mask]
valid_fpr        = fpr[valid_mask]
best_threshold   = float(valid_thresholds[np.argmin(valid_fpr)])
print(f'Selected threshold: {best_threshold:.4f}')
np.save('best_threshold.npy', np.array([best_threshold]))

Selected threshold: 0.8943


After getting the best threshold and validating the performance in each fold, we build the final training model

In [ ]:
full_train_ds     = HeartbeatDataset(X_train_norm, y_train, train_patient_ids)
full_train_loader = DataLoader(full_train_ds, batch_size = BATCH_SIZE, shuffle = True)

final_model = MurmurCNN1D().to(device)
optimizer   = torch.optim.Adam(final_model.parameters(), lr = LEARNING_RATE)
pos_weight  = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()], dtype=torch.float32).to(device)
criterion   = nn.BCEWithLogitsLoss(pos_weight = pos_weight)

for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch(final_model, full_train_loader, criterion, optimizer)
    if (epoch + 1) % 5 == 0:
        print(f'  Epoch {epoch+1}/{NUM_EPOCHS} — Loss: {train_loss:.4f}')

print('Final model training complete.')

torch.save(final_model.state_dict(), 'model1_weights.pth')

  Epoch 5/40 — Loss: 0.8439
  Epoch 10/40 — Loss: 0.7880
  Epoch 15/40 — Loss: 0.7389
  Epoch 20/40 — Loss: 0.6949
  Epoch 25/40 — Loss: 0.6562
  Epoch 30/40 — Loss: 0.6148
  Epoch 35/40 — Loss: 0.5815
  Epoch 40/40 — Loss: 0.5431
Final model training complete.


### FInal Model Eval

We create a test dataset and dataloader using the normalized test data
Use the final trained model to get all the preidctions for all the test samples
Aggregate them to the pateint level
Get metrics with the selected threshold

In [25]:
final_model = MurmurCNN1D().to(device)
final_model.load_state_dict(torch.load('model1_weights.pth', map_location=device))
final_model.eval()

test_dataset = HeartbeatDataset(X_test_norm, y_test, test_patient_ids)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_probs, test_labels, test_pids_out = get_heartbeat_predictions(final_model, test_loader)
test_patient_df = aggregate_to_patient_level(test_probs, test_labels, test_pids_out)
test_metrics = compute_metrics(test_patient_df['label'], test_patient_df['pred'], best_threshold)
print(f'Test set metrics at threshold {best_threshold:.4f}:')
for metric, value in test_metrics.items():
    print(f'  {metric}: {value:.4f}')

test_patient_df.to_csv('model1_test_predictions.csv', index=False)
metrics_df = pd.DataFrame([test_metrics])
metrics_df['model'] = 'Model1_1DCNN'
metrics_df.to_csv('model1_metrics.csv', index=False)

Test set metrics at threshold 0.8943:
  auc: 0.7398
  auprc: 0.5550
  f1: 0.5424
  sensitivity: 0.4444
  specificity: 0.9496
  threshold: 0.8943


## Zone Specific Analysis
Evaluating each location seperately

In [26]:
print ('Zone Specific Murmur Detection:')

for location in np.unique(train_locations):
    loc_mask = test_locations == location #create mask for the location in the test set
    
    loc_probs = test_probs[loc_mask]
    loc_labels = test_labels[loc_mask]
    loc_patient_ids = test_pids_out[loc_mask]

    #aggregate to patient level
    loc_patient_df = aggregate_to_patient_level(loc_probs, loc_labels, loc_patient_ids)

    # compute metrics
    loc_metrics = compute_metrics(loc_patient_df['label'], loc_patient_df['pred'], best_threshold)
    
    print(f'\n{location}:')
    for metric, value in loc_metrics.items():
        print(f'  {metric}: {value:.4f}')

Zone Specific Murmur Detection:

AV:
  auc: 0.7710
  auprc: 0.6114
  f1: 0.4737
  sensitivity: 0.3103
  specificity: 1.0000
  threshold: 0.8943

MV:
  auc: 0.7309
  auprc: 0.4863
  f1: 0.3333
  sensitivity: 0.2121
  specificity: 0.9841
  threshold: 0.8943

PV:
  auc: 0.7722
  auprc: 0.6213
  f1: 0.5116
  sensitivity: 0.3793
  specificity: 0.9744
  threshold: 0.8943

TV:
  auc: 0.6954
  auprc: 0.5836
  f1: 0.5641
  sensitivity: 0.4231
  specificity: 0.9821
  threshold: 0.8943
